# Pipeline — Patents, Papers, Parts & Planet

This single notebook runs the full data pipeline from ingestion to website export.
Each cell runs one script from the `scripts/` directory and shows its output.

You can also run any step on its own from the terminal:
```bash
python scripts/01_ingest_papers.py
```

## Steps
| Step | Script | What it does |
|------|--------|--------------|
| 1 | `01_ingest_papers.py` | Download papers from OpenAlex |
| 2 | `02_ingest_patents.py` | Download patents from Lens.org |
| 3 | `03_ingest_projects.py` | Load iGEM projects and parts from CSV |
| 4 | `04_embed.py` | Generate sentence embeddings |
| 5 | `05_cluster.py` | UMAP projection + HDBSCAN clustering |
| 6 | `06_visualize.py` | Export JSON files for the website |

After step 6, a **Case Study section** below explores the carbon-capture subset.

---
**Before running:**
- Copy `.env.example` to `.env` and fill in your API keys
- Place iGEM CSV files in `data/raw/projects/` and `data/raw/parts/`
- Install dependencies: `pip install -r requirements.txt`

In [ ]:
import sys
from pathlib import Path

# Add repo root to path so scripts can import src/
REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

print(f'Working from: {REPO_ROOT}')

## Step 1 — Ingest Papers

Builds the synthetic biology paper corpus in three layers:

1. **Core keywords** — `'synthetic biology'`, `'synthetic genomics'`, `'synthetic genome'`
   High-precision retrieval; these terms almost always mean synthetic biology.
2. **Subfield keywords** — `'BioBrick'`, `'repressilator'`, `'minimal genome'`, etc.
   Catches foundational work that predates or avoids the 'synthetic biology' label.
3. **Citation expansion** — backward and forward snowballing from Layer 1 seed papers.
   Captures related work using different terminology.

Each paper is tagged with a `retrieval_reason` field recording which layer found it.
Broad terms like `'metabolic engineering'` are **not** used for retrieval — they would
swamp the corpus. They appear only as post-hoc classification labels.

Strategy: Shapira, Kwon & Youtie (2017, *Scientometrics* 112(3), 1457–1478).

In [ ]:
%run ../scripts/01_ingest_papers.py

## Step 2 — Ingest Patents

Retrieves synthetic biology patents from Lens.org using a **combined IPC classification
+ keyword strategy**, following van Doren, Koenigstein & Reiss (2013).

### Why keywords alone are not enough

Synthetic biology has no dedicated IPC or CPC patent classification code — patents are
scattered across C12N (genetic engineering), C12P (fermentation), C12Q (diagnostic
nucleic acids), C12S (processes using enzymes), and C40B (combinatorial chemistry /
gene libraries). A pure keyword search on terms like "synthetic biology" dramatically
overestimates activity because the field's terminology overlaps heavily with general
biotechnology and genetic engineering (Oldham & Hall, 2018).

### Strategy: IPC codes + keywords (AND logic)

Van Doren et al. (2013) retrieved 1,195 patents (1990–2010) using a three-part approach:

1. **IPC scope filter** — restrict to the relevant technical classes:
   `C12N`, `C12P`, `C12Q`, `C12S`, `C40B`
2. **Knowledge & engineering keywords** — `"synthetic biology"`, `"genetic circuit"`,
   `"synthetic genome"`, `"DNA assembly"`, `"BioBrick"`
3. **Enabling technology keywords** — `"gene synthesis"`, `"DNA synthesis"`,
   `"cell-free"`, `"metabolic pathway engineering"`

Combining the IPC filter (AND) with keywords means we only retrieve patents that are
both in the right technical class *and* mention synthetic biology concepts — reducing
false positives from general biotech.

### References
- van Doren, D., Koenigstein, S., & Reiss, T. (2013). The development of synthetic
  biology: a patent analysis. *Systems and Synthetic Biology*, 7(4), 209–220.
  https://doi.org/10.1007/s11693-013-9121-7
- Oldham, P., & Hall, S. (2018). Synthetic Biology: Mapping the Patent Landscape.
  *bioRxiv*. https://doi.org/10.1101/483826

In [ ]:
%run ../scripts/02_ingest_patents.py

## Step 3 — Ingest iGEM Projects and Parts

Loads local CSV files downloaded from the iGEM Registry and saves
`data/processed/projects.csv` and `data/processed/parts.csv`.

Place iGEM data in:
- `data/raw/projects/igem_projects.csv`
- `data/raw/parts/igem_parts.csv`

In [ ]:
%run ../scripts/03_ingest_projects.py

## Step 4 — Generate Embeddings

Encodes every artifact's `text` field as a 768-dimensional dense vector using
**SPECTER** (`allenai-specter`), a transformer model trained on scientific papers
using citation-informed contrastive learning. SPECTER is better suited than
general-purpose sentence models for capturing semantic relatedness across patents,
papers, and student projects in scientific domains like synthetic biology. It loads
cleanly via `sentence-transformers` with no extra dependencies.

Reference: Cohan et al. (2020) "SPECTER: Document-level Representation Learning
using Citation-informed Transformers." ACL 2020. https://arxiv.org/abs/2004.13313

Results are cached in `data/embeddings/embeddings.json`. Already-encoded items
are skipped, so this is safe to re-run.

> **Note:** The first run downloads ~440 MB of model weights. If you switch
> embedding models, delete `data/embeddings/embeddings.json` first so the cache
> is rebuilt — vectors from different models are not compatible.

In [15]:
%run ../scripts/04_embed.py

=== Step 4: Generate Embeddings ===



INFO  Detected SPECTER2 — loading with adapters library


Loaded 9466 records from papers.csv
Empty: patents.csv (skipping)
Loaded 4615 records from projects.csv

Total: 14081 artifacts across 2 types

Loading model: allenai/specter2_base
(First run downloads ~440 MB of model weights — this is normal.)


ModuleNotFoundError: No module named 'adapters'

## Step 5 — UMAP Projection and Clustering

Projects the high-dimensional embeddings to 2D with UMAP, then finds thematic
clusters with HDBSCAN. Noise points (no cluster found) get label `-1`.

In [ ]:
%run ../scripts/05_cluster.py

## Step 6 — Export for Website

Writes the three JSON files that the website's interactive visualizations load:
- `artifacts.json` — metadata for every artifact
- `projections.json` — UMAP coordinates
- `cities.json` — city-level artifact counts

In [ ]:
%run ../scripts/06_visualize.py

---

# Case Study — Carbon Capture in Synthetic Biology

This section explores the carbon-capture subset of the corpus in detail.
It is designed to be readable as a standalone reproducibility companion on the website.

**What we look at:**
1. How many artifacts were tagged as carbon-capture, and of what types?
2. Where do carbon-capture artifacts sit in the shared semantic space?
3. How has carbon-capture activity changed over time?
4. Which cities show the most carbon-capture activity?

In [ ]:
import json
import pandas as pd
import plotly.express as px
from pathlib import Path

PROCESSED   = REPO_ROOT / 'data' / 'processed'
EMBEDDINGS  = REPO_ROOT / 'data' / 'embeddings'

In [ ]:
# --- Load all processed datasets ---

dfs = {}
for name in ['papers', 'patents', 'projects', 'parts']:
    path = PROCESSED / f'{name}.csv'
    if path.exists():
        dfs[name] = pd.read_csv(path)
        print(f'{name}: {len(dfs[name])} records')

all_artifacts = pd.concat(list(dfs.values()), ignore_index=True)
print(f'\nTotal: {len(all_artifacts)} artifacts')

In [ ]:
# --- Filter to carbon-capture subset ---

cc = all_artifacts[all_artifacts['case_study_flag'] == True].copy()

print(f'Carbon capture artifacts: {len(cc)}')
print()
print(cc['type'].value_counts().to_string())

In [ ]:
# --- Load UMAP projections and merge ---

proj_path = EMBEDDINGS / 'projections.json'
if proj_path.exists():
    with open(proj_path) as f:
        projections = pd.DataFrame(json.load(f))
    all_proj = all_artifacts.merge(projections, on='id', how='inner')
    cc_proj  = cc.merge(projections, on='id', how='inner')
    print(f'Projections loaded: {len(projections)} entries')
else:
    print('projections.json not found — run Step 5 first.')
    all_proj = cc_proj = None

In [ ]:
# --- Semantic space: all artifacts, carbon capture highlighted ---

if all_proj is not None:
    all_proj['highlight'] = all_proj['id'].isin(cc['id'])
    all_proj['label'] = all_proj['highlight'].map(
        {True: 'Carbon capture', False: 'Other'}
    )

    fig = px.scatter(
        all_proj,
        x='x', y='y',
        color='label',
        color_discrete_map={'Carbon capture': '#e15759', 'Other': '#cccccc'},
        opacity=0.6,
        hover_data=['title', 'type', 'year'],
        title='Semantic Space — Carbon Capture Highlighted',
        labels={'x': 'UMAP 1', 'y': 'UMAP 2'},
    )
    fig.update_traces(marker_size=4)
    fig.update_layout(legend_title_text='')
    fig.show()

In [ ]:
# --- Semantic space: carbon-capture artifacts only, coloured by type ---

if cc_proj is not None and len(cc_proj) > 0:
    fig = px.scatter(
        cc_proj,
        x='x', y='y',
        color='type',
        hover_data=['title', 'year', 'city'],
        title='Carbon Capture Artifacts — Semantic Space by Type',
        labels={'x': 'UMAP 1', 'y': 'UMAP 2'},
    )
    fig.update_traces(marker_size=6)
    fig.show()

In [ ]:
# --- Timeline: carbon capture activity by year and artifact type ---

cc_by_year = (
    cc.groupby(['year', 'type'])
    .size()
    .reset_index(name='count')
)

fig = px.bar(
    cc_by_year,
    x='year', y='count', color='type',
    barmode='stack',
    title='Carbon Capture Activity by Year and Artifact Type',
    labels={'year': 'Year', 'count': 'Number of Artifacts'},
)
fig.show()

In [ ]:
# --- City-level summary ---

cc_cities = (
    cc.groupby(['city', 'country', 'type'])
    .size()
    .reset_index(name='count')
    .pivot_table(index=['city', 'country'], columns='type', values='count', fill_value=0)
    .reset_index()
)
cc_cities['total'] = cc_cities.drop(columns=['city', 'country'], errors='ignore').sum(axis=1)

print('Top cities for carbon capture activity:')
print(cc_cities.sort_values('total', ascending=False).head(15).to_string(index=False))